In [2]:
# 导包
from crewai.tools import tool  # 这个有时候版本不兼容会报错
# from langchain.tools import tool  # 这个可以
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import smtplib


In [1]:
1+1

2

In [3]:
"""
2个函数。
函数1：将文本写入并保存为txt文件
函数2：寄信，将txt文件发送到指定邮箱
"""
load_dotenv()
PASSWORD = os.getenv("qq_2765_authcode")
@tool
def store_poem_to_txt(content:str) -> str:
    """
    将编辑后的书信文本内容自动保存到txt文档中
    """
    try:
        filename = "情书.txt"
        with open(filename, 'w', encoding="utf-8") as file:
            file.write(content)
            print(f"文件已经写入{filename}文件")
    except Exception as e:
        return f"文件写入失败,原因：{e}"
@tool
def send_email() -> str:
    """
    将编辑后的书信文本内容自动发送到指定邮箱
    """
    sender_email = "2765338781@qq.com"
    sender_password = PASSWORD
    receiver_email = "2765338781@qq.com"
    subject = "一封情书❤"
    with open("情书.txt", encoding="utf-8") as f:
        body = f.read()
    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))
    try:
        # 连接smtp服务器并发送邮件
        # 创建一个安全的ssl连接
        smtp_server = smtplib.SMTP_SSL('smtp.qq.com', 465)
        # 建立连接
        smtp_server.connect('smtp.qq.com', 465)
        # 登录邮箱(传入发送人的邮箱名和授权码)
        smtp_server.login(sender_email, sender_password)
        # 发送邮件
        smtp_server.sendmail(sender_email, receiver_email, msg.as_string())
        print("邮件发送成功!")
    except Exception as e:
        print(f"邮件发送失败!原因：{e}")
    finally:
        smtp_server.quit()


In [5]:
from crewai import Agent, LLM, Task, Crew, Process
import os
from dotenv import load_dotenv
load_dotenv()
ds_api_key = os.getenv("ds_api_key")
ds_base_url = os.getenv("ds_base_url")
# TODO 0.提前定义好LLM大语言模型

llm_deepseek = LLM(
    model="deepseek-chat",
    api_key=ds_api_key,
    base_url=ds_base_url
)
# TODO 1.创建智能体
# 1.1 创建作家智能体
writer = Agent(
    role="作家",
    goal="根据用户的需求,创作出情感比较丰富的文字(要求字数在300左右)",
    backstory="你作为一个非常优秀的作家,拥有丰富的写作经验,最擅长编写情感丰富的文章",
    verbose=True,
    allow_delegation=False,
    llm=llm_deepseek
)
# 1.2 创建编辑者智能体
editor = Agent(
    role="编辑者",
    goal="请你对上面作家创作的文章进行编辑优化,让内容更加丰富,编辑成标准的书信格式",
    backstory="""你作为一个非常专业的编辑,你在编辑书信方面有很多年的经验,你需要将文章编写成书信格式,
            并将编辑好的书信内容存储到本地磁盘上,你必须使用tools列表中的store_poem_to_txt工具
            将编辑好的书信内容存储到指定文件中
    """,
    tools=[store_poem_to_txt],
    verbose=True,
    allow_delegation=False,
    llm=llm_deepseek
)
# 1.3 创建编辑者智能体
sender = Agent(
    role="寄信者",
    goal="请你把上面编辑者编辑并优化好的书信以邮件的形式发送给用户指定的收件人,必须保证发送成功,再结束!!!",
    backstory="""你作为一个非常专业的寄信者,在发邮件方面有多年的经验,请你把上面编辑者
        优化好的情书发送给指定收件人,你必须使用tools中的send_email工具将书信发送到指定邮箱
    """,
    tools=[send_email],
    verbose=True,
    allow_delegation=False,
    llm=llm_deepseek
)
# TODO 2.构建任务,指定智能体
# todo 先获取用户的需求
user_input = input("请您输入您写情书的要求和收件人:")
# 2.1 写情书
task1 = Task(
    description=f"根据用户的需求:{user_input},生成一个非常浪漫真挚的情书",
    expected_output="已经生成一份情感真挚的情书(300字左右)",
    agent=writer
)
# 2.2 编辑情书为书信格式
task2 = Task(
    description=f"对上述写好的情书进行编辑优化,使其更加丰富,符合书信格式",
    expected_output="已经编辑并优化好情书为书信格式,并且成功保存到本地磁盘上",
    agent=editor
)
# 2.3 编辑情书为书信格式
task3 = Task(
    description=f"把优化为书信格式的情书,进行邮件发送到指定邮箱,这个很重要!必须要发送成功",
    expected_output="发送邮件任务已经调用成功,请查看邮箱",
    agent=sender
)
# TODO 3.创建Crew对象,传入智能体和任务并触发运行
crew = Crew(
    agents=[writer, editor, sender],
    tasks=[task1, task2, task3],
    verbose=True,
    Process=Process.sequential  # 当前是顺序执行,后续如果需要并行执行,请修改为Process.parallel
)
print('------------------------------------------')
# todo 触发执行
result = crew.kickoff()
print(result)



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

------------------------------------------


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7c67962e-e779-4b32-9eca-654edddebc91                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



🚀 Crew: crew
└── 📋 Task: 92de106d-825a-4cad-9cd5-cf14812724a1
       Status: Executing Task...


Provider List: https://docs.litellm.ai/docs/providers



...

🚀 Crew: crew
├── 📋 Task: 92de106d-825a-4cad-9cd5-cf14812724a1
│      Assigned to: 作家
│      Status: ✅ Completed
│   └── 🤖 Agent: 作家
│           Status: ✅ Completed
├── 📋 Task: 98d03e92-2696-42f2-a6ac-28ca1f2a2a14
│      Assigned to: 编辑者
│      Status: ✅ Completed
│   └── 🤖 Agent: 编辑者
│           Status: ✅ Completed
└── 📋 Task: 308d0e85-7318-4f09-9992-6e289dbd02ca
       Status: Executing Task...
    └── 🤖 Agent: 寄信者
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: 92de106d-825a-4cad-9cd5-cf14812724a1
│      Assigned to: 作家
│      Status: ✅ Completed
│   └── 🤖 Agent: 作家
│           Status: ✅ Completed
├── 📋 Task: 98d03e92-2696-42f2-a6ac-28ca1f2a2a14
│      Assigned to: 编辑者
│      Status: ✅ Completed
│   └── 🤖 Agent: 编辑者
│           Status: ✅ Completed
└── 📋 Task: 308d0e85-7318-4f09-9992-6e289dbd02ca
       Assigned to: 寄信者
       Status: ✅ Completed
    └── 🤖 Agent: 寄信者
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 308d0e85-7318-4f09-9992-6e289dbd02ca                                                                     │
│  Agent: 寄信者                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7c67962e-e779-4b32-9eca-654edddebc91                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

发送邮件任务已经调用成功,请查看邮箱
